# LeetCode 1262 — Greatest Sum Divisible by Three bằng Python

Link đề: https://leetcode.com/problems/greatest-sum-divisible-by-three/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxSumDivThree(self, nums: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết nghĩ theo **số dư** (modulo): chỉ phần dư khi chia 3 của tổng quyết định chia hết hay không.
2. Đi từ **thử mọi tập con** `O(2^n)` → **tham lam: lấy hết rồi bỏ ít nhất** → **DP theo số dư** `dp[r]` `O(n)`.
3. Thành thạo kỹ thuật DP "trạng thái nhỏ" (3 trạng thái) và **cập nhật dùng bản sao** để không dùng một phần tử hai lần.
4. Hiểu vì sao khởi tạo trạng thái không đạt được bằng `-inf`, và tổng quát hoá cho chia hết cho `k`.
5. Tránh lỗi: chỉ xét bỏ **một** phần tử, cập nhật `dp` tại chỗ, khởi tạo `dp = [0, 0, 0]`, gọi `min` trên danh sách rỗng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums`, trả về **tổng lớn nhất** của một tập các phần tử (chọn tuỳ ý, mỗi phần tử tối đa một lần, có thể không chọn gì) sao cho tổng **chia hết cho 3**.

**Ví dụ 1:** `nums = [3,6,5,1,8]` → `18` (chọn `3, 6, 1, 8`).

**Ví dụ 2:** `nums = [4]` → `0` (không chọn gì; tổng rỗng = 0 chia hết cho 3).

**Ví dụ 3:** `nums = [1,2,3,4,4]` → `12` (chọn `1, 3, 4, 4`).

**Ràng buộc:** `1 <= nums.length <= 4·10^4`, `1 <= nums[i] <= 10^4`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import combinations
from typing import List

examples = [([3, 6, 5, 1, 8], 18), ([4], 0), ([1, 2, 3, 4, 4], 12)]
for nums, want in examples:
    print(nums, "->", want, "| tổng toàn bộ =", sum(nums), "dư", sum(nums) % 3)

[3, 6, 5, 1, 8] -> 18 | tổng toàn bộ = 23 dư 2
[4] -> 0 | tổng toàn bộ = 4 dư 1
[1, 2, 3, 4, 4] -> 12 | tổng toàn bộ = 14 dư 2


## 3. Kiến thức nền

### 3.1 Phép chia lấy dư `%`

`a % 3` ∈ {0, 1, 2}. Tính chất quan trọng: `(a + b) % 3 == (a % 3 + b % 3) % 3`. Nên muốn biết tổng chia hết cho 3 hay không, chỉ cần theo dõi **tổng các số dư**.

In [2]:
a, b = 7, 11
print(a % 3, b % 3, (a + b) % 3, (a % 3 + b % 3) % 3)
print([x % 3 for x in [3, 6, 5, 1, 8]])

1 2 0 0
[0, 0, 2, 1, 2]


### 3.2 Liệt kê tập con

Có `2^n` tập con. Cách đơn giản: duyệt `mask` từ `0` đến `2^n − 1`, bit `i` bật ⇒ chọn `nums[i]`. Hoặc dùng `itertools.combinations` với mọi kích thước.

In [3]:
nums = [1, 2, 3]
for mask in range(1 << len(nums)):
    chosen = [nums[i] for i in range(len(nums)) if mask >> i & 1]
    print(f"mask={mask:03b} -> {chosen}, tổng={sum(chosen)}")

mask=000 -> [], tổng=0
mask=001 -> [1], tổng=1
mask=010 -> [2], tổng=2
mask=011 -> [1, 2], tổng=3
mask=100 -> [3], tổng=3
mask=101 -> [1, 3], tổng=4
mask=110 -> [2, 3], tổng=5
mask=111 -> [1, 2, 3], tổng=6


### 3.3 Sắp xếp và lọc theo số dư

`sorted(x for x in nums if x % 3 == 1)` cho các số dư 1 theo thứ tự tăng ⇒ hai phần tử đầu là hai số **nhỏ nhất** dư 1.

In [4]:
nums = [3, 6, 5, 1, 8, 4, 7]
r1 = sorted(x for x in nums if x % 3 == 1)
r2 = sorted(x for x in nums if x % 3 == 2)
print("dư 1:", r1, "| dư 2:", r2)
print("nhỏ nhất dư 1:", r1[:1], "| hai nhỏ nhất dư 2:", r2[:2])

dư 1: [1, 4, 7] | dư 2: [5, 8]
nhỏ nhất dư 1: [1] | hai nhỏ nhất dư 2: [5, 8]


### 3.4 Vô cực âm `float('-inf')`

Trong DP "tìm max", trạng thái **chưa đạt được** nên mang giá trị `-inf`: cộng thêm bao nhiêu cũng vẫn `-inf`, và `max(-inf, x) = x`. Nếu dùng `0` thay cho `-inf`, ta sẽ ngầm giả vờ có một tập đạt trạng thái đó với tổng 0 — sai.

In [5]:
NEG = float("-inf")
print(NEG + 100, max(NEG, 5), max(NEG, NEG + 7))

-inf 5 -inf


### 3.5 Sao chép danh sách nhỏ: `dp[:]`

Khi cập nhật DP theo từng phần tử, ta tính mảng mới từ mảng **cũ**. `new = dp[:]` tạo bản sao; sửa `new` không làm đổi `dp`.

In [6]:
dp = [0, NEG, NEG]
new = dp[:]
new[1] = 10
print("dp =", dp, "| new =", new)

dp = [0, -inf, -inf] | new = [0, 10, -inf]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `[3, 6, 5, 1, 8]`:

- Tổng toàn bộ `= 23`, `23 % 3 = 2` ⇒ chưa chia hết.
- Muốn tổng lớn nhất ⇒ **lấy hết rồi bỏ đi càng ít càng tốt** để phần dư về 0.
- Dư 2 ⇒ bỏ đi một lượng có dư 2. Hai lựa chọn:
  - bỏ **một** số dư 2: các số dư 2 là `5, 8` ⇒ bỏ `5` (nhỏ nhất) ⇒ `23 − 5 = 18`;
  - bỏ **hai** số dư 1 (1 + 1 = 2): số dư 1 chỉ có `1` ⇒ không đủ.
- Đáp án `18` ✔.

**Ví dụ 3** `[1, 2, 3, 4, 4]`: tổng `14`, dư 2. Bỏ một số dư 2: `2` ⇒ `12`. Bỏ hai số dư 1: `1 + 4 = 5` ⇒ `9`. Lấy max ⇒ `12` ✔.

In [7]:
for nums, want in examples:
    s = sum(nums)
    r1 = sorted(x for x in nums if x % 3 == 1)
    r2 = sorted(x for x in nums if x % 3 == 2)
    print(f"{nums}: tổng={s}, dư={s % 3}, dư1={r1}, dư2={r2} -> đáp án {want}")

[3, 6, 5, 1, 8]: tổng=23, dư=2, dư1=[1], dư2=[5, 8] -> đáp án 18
[4]: tổng=4, dư=1, dư1=[4], dư2=[] -> đáp án 0
[1, 2, 3, 4, 4]: tổng=14, dư=2, dư1=[1, 4, 4], dư2=[2] -> đáp án 12


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi tập con, giữ tổng lớn nhất chia hết cho 3.

In [8]:
def max_sum_brute(nums):
    n = len(nums)
    best = 0
    for mask in range(1 << n):
        s = 0
        for i in range(n):
            if mask >> i & 1:
                s += nums[i]
        if s % 3 == 0 and s > best:
            best = s
    return best

for nums, want in examples:
    print(max_sum_brute(nums), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0
12 | mong đợi 12


**Độ phức tạp:** `O(2^n · n)` thời gian, `O(1)` bộ nhớ. Với `n = 40` đã là `4·10^13` — vô vọng với `n = 4·10^4`.

**Nút thắt:** ta phân biệt các tập con theo **từng phần tử**, trong khi điều duy nhất quan trọng là **tổng dư mấy**. Hai tập con có cùng phần dư thì chỉ cần giữ tập có tổng lớn hơn ⇒ chỉ còn 3 "đại diện".

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tham lam):** đáp án = tổng toàn bộ − (lượng nhỏ nhất phải bỏ). Nếu `sum % 3 == 1`, lượng bỏ đi phải có dư 1: hoặc **một số dư 1** nhỏ nhất, hoặc **hai số dư 2** nhỏ nhất (2 + 2 = 4 ≡ 1). Tương tự với dư 2: một số dư 2, hoặc hai số dư 1.

Tại sao không cần bỏ nhiều hơn? Bỏ ba số dư 2 (≡ 0) là thừa — giữ lại ba số đó không làm đổi phần dư mà tổng lớn hơn. Lập luận đầy đủ ở mục 10.

**Insight 2 (DP):** `dp[r]` = tổng lớn nhất của một tập con (trong các phần tử đã xét) có `tổng % 3 == r`. Thêm phần tử `x`:

> `new[(r + x) % 3] = max(new[(r + x) % 3], dp[r] + x)` với mọi `r`.

**Quy tắc "nếu… thì…":**
- Nếu tổng toàn bộ đã chia hết ⇒ trả luôn.
- Nếu dư 1 ⇒ bỏ `min(một số dư 1 nhỏ nhất, hai số dư 2 nhỏ nhất)`.
- Nếu dư 2 ⇒ bỏ `min(một số dư 2 nhỏ nhất, hai số dư 1 nhỏ nhất)`.
- Trạng thái DP chưa đạt ⇒ `-inf`.

**Invariant DP:** sau khi xét `i` phần tử đầu, `dp[r]` đúng bằng max tổng các tập con của `nums[:i]` có dư `r` (hoặc `-inf` nếu không có).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam: lấy hết rồi bỏ ít nhất (sắp xếp)

1. `s = sum(nums)`; nếu `s % 3 == 0` trả `s`.
2. Lọc và sắp các số dư 1, dư 2.
3. Tính các phương án bỏ hợp lệ (đủ phần tử), lấy phương án nhỏ nhất. Luôn có ít nhất một phương án hợp lệ (chứng minh ngay dưới).

In [9]:
def max_sum_greedy_sort(nums):
    s = sum(nums)
    r = s % 3
    if r == 0:
        return s
    ones = sorted(x for x in nums if x % 3 == 1)
    twos = sorted(x for x in nums if x % 3 == 2)
    single, pair = (ones, twos) if r == 1 else (twos, ones)
    candidates = []
    if len(single) >= 1:
        candidates.append(single[0])
    if len(pair) >= 2:
        candidates.append(pair[0] + pair[1])
    return s - min(candidates)

for nums, want in examples:
    print(max_sum_greedy_sort(nums), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0
12 | mong đợi 12


Vì sao `candidates` không bao giờ rỗng khi `r != 0`? Nếu `r == 1` mà không có số dư 1 nào thì tổng dư của tất cả là `2·(số lượng số dư 2) ≡ 1 (mod 3)` ⇒ số lượng số dư 2 ≡ 2 (mod 3) ⇒ có ít nhất 2 số dư 2. Tương tự cho `r == 2`.

**Độ phức tạp Cách 2:** `O(n log n)` (sắp xếp) thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Tham lam `O(n)`: chỉ giữ hai số nhỏ nhất mỗi loại

Không cần sắp toàn bộ: chỉ cần **2 số nhỏ nhất dư 1** và **2 số nhỏ nhất dư 2**. Duy trì bằng biến (hoặc `heapq.nsmallest(2, ...)`).

In [10]:
def max_sum_greedy_linear(nums):
    INF = float("inf")
    s = 0
    a1 = a2 = INF          # hai số nhỏ nhất dư 1 (a1 <= a2)
    b1 = b2 = INF          # hai số nhỏ nhất dư 2 (b1 <= b2)
    for x in nums:
        s += x
        if x % 3 == 1:
            if x < a1:
                a1, a2 = x, a1
            elif x < a2:
                a2 = x
        elif x % 3 == 2:
            if x < b1:
                b1, b2 = x, b1
            elif x < b2:
                b2 = x
    if s % 3 == 0:
        return s
    if s % 3 == 1:
        return s - min(a1, b1 + b2)
    return s - min(b1, a1 + a2)

for nums, want in examples:
    print(max_sum_greedy_linear(nums), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0
12 | mong đợi 12


Bản nộp LeetCode của Cách 3:

In [11]:
class SolutionGreedy:
    def maxSumDivThree(self, nums: list[int]) -> int:
        INF = float("inf")
        s = 0
        a1 = a2 = b1 = b2 = INF
        for x in nums:
            s += x
            if x % 3 == 1:
                if x < a1:
                    a1, a2 = x, a1
                elif x < a2:
                    a2 = x
            elif x % 3 == 2:
                if x < b1:
                    b1, b2 = x, b1
                elif x < b2:
                    b2 = x
        r = s % 3
        if r == 0:
            return s
        drop = min(a1, b1 + b2) if r == 1 else min(b1, a1 + a2)
        return s - drop

print([SolutionGreedy().maxSumDivThree(n) for n, _ in examples])

[18, 0, 12]


**Độ phức tạp Cách 3:** **`O(n)` thời gian, `O(1)` bộ nhớ**. Lưu ý: `INF` là `float`, nhưng phương án chọn được luôn hữu hạn (đã chứng minh), nên kết quả trả về là số nguyên.

### 7.3 Cách 4 — DP bảng `dp[i][r]`

`dp[i][r]` = tổng lớn nhất dùng `i` phần tử đầu, dư `r`. Chuyển trạng thái: với phần tử `x = nums[i−1]`, hoặc **không chọn** (`dp[i−1][r]`), hoặc **chọn** (`dp[i−1][(r − x) % 3] + x`).

In [12]:
def max_sum_dp_table(nums):
    NEG = float("-inf")
    n = len(nums)
    dp = [[NEG] * 3 for _ in range(n + 1)]
    dp[0][0] = 0                      # tập rỗng: tổng 0, dư 0
    for i in range(1, n + 1):
        x = nums[i - 1]
        for r in range(3):
            skip = dp[i - 1][r]
            take = dp[i - 1][(r - x) % 3] + x
            dp[i][r] = max(skip, take)
    return dp[n][0]

for nums, want in examples:
    print(max_sum_dp_table(nums), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0
12 | mong đợi 12


**Độ phức tạp Cách 4:** `O(3n) = O(n)` thời gian, `O(3n)` bộ nhớ.

### 7.4 Cách 5 — DP thu gọn `dp[3]` (lời giải phổ biến nhất)

Chỉ cần hàng trước ⇒ giữ mảng 3 phần tử. Mỗi phần tử: `new = dp[:]`, rồi với mỗi `r`, cập nhật `new[(r + x) % 3]` từ `dp[r] + x`.

In [13]:
def max_sum_dp(nums):
    NEG = float("-inf")
    dp = [0, NEG, NEG]
    for x in nums:
        new = dp[:]
        for r in range(3):
            nr = (r + x) % 3
            new[nr] = max(new[nr], dp[r] + x)
        dp = new
    return dp[0]

for nums, want in examples:
    print(max_sum_dp(nums), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0
12 | mong đợi 12


Bản nộp LeetCode của Cách 5 (đây cũng là bản chính ở mục 9). Một biến thể hay gặp: khởi tạo `dp = [0, 0, 0]` rồi chỉ cập nhật khi `dp[r] % 3 == r` hoặc `dp[r] > 0` — cách đó đúng nhưng dễ nhầm; dùng `-inf` rõ ràng hơn.

In [14]:
class SolutionDP:
    def maxSumDivThree(self, nums: list[int]) -> int:
        dp = [0, float("-inf"), float("-inf")]
        for x in nums:
            new = dp[:]
            for r in range(3):
                nr = (r + x) % 3
                if dp[r] + x > new[nr]:
                    new[nr] = dp[r] + x
            dp = new
        return dp[0]

print([SolutionDP().maxSumDivThree(n) for n, _ in examples])

[18, 0, 12]


**Độ phức tạp Cách 5:** **`O(n)` thời gian, `O(1)` bộ nhớ**.

**(Thêm) Đệ quy có nhớ** theo gợi ý của đề `DP[pos][mod]`: `f(i, m)` = tổng lớn nhất có thể thêm từ vị trí `i` khi tổng hiện tại dư `m`. Đẹp về lý thuyết, nhưng độ sâu đệ quy `n = 4·10^4` sẽ vượt giới hạn của Python ⇒ chỉ dùng cho mảng nhỏ (dưới đây dùng để đối chiếu).

In [15]:
def max_sum_memo(nums):
    n = len(nums)
    @lru_cache(maxsize=None)
    def f(i, m):
        if i == n:
            return 0 if m == 0 else float("-inf")
        return max(f(i + 1, m), nums[i] + f(i + 1, (m + nums[i]) % 3))
    return f(0, 0)

for nums, want in examples:
    print(max_sum_memo(nums), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0
12 | mong đợi 12


## 8. TRACE từng bước

Trace **Cách 5** với `[3, 6, 5, 1, 8]`:

| `x` | `x % 3` | `dp` trước | `dp` sau | Giải thích |
|---|---|---|---|---|
| 3 | 0 | `[0, -∞, -∞]` | `[3, -∞, -∞]` | thêm 3 vào tập dư 0 |
| 6 | 0 | `[3, -∞, -∞]` | `[9, -∞, -∞]` | |
| 5 | 2 | `[9, -∞, -∞]` | `[9, -∞, 14]` | 9+5=14 dư 2 |
| 1 | 1 | `[9, -∞, 14]` | `[15, 10, 14]` | 14+1=15 dư 0; 9+1=10 dư 1 |
| 8 | 2 | `[15, 10, 14]` | `[18, 22, 23]` | 10+8=18 dư 0; 14+8=22 dư 1; 15+8=23 dư 2 |

Đáp án `dp[0] = 18`.

In [16]:
def trace_dp(nums):
    NEG = float("-inf")
    fmt = lambda d: "[" + ", ".join("-∞" if v == NEG else str(v) for v in d) + "]"
    dp = [0, NEG, NEG]
    print(f"khởi tạo: dp = {fmt(dp)}")
    for x in nums:
        new = dp[:]
        changes = []
        for r in range(3):
            nr = (r + x) % 3
            if dp[r] + x > new[nr]:
                new[nr] = dp[r] + x
                changes.append(f"dp[{r}]+{x}={dp[r] + x} -> new[{nr}]")
        dp = new
        print(f"x={x:2} (dư {x % 3}): {', '.join(changes) or 'không đổi'}  => dp = {fmt(dp)}")
    print("Đáp án:", dp[0])
    return dp[0]

trace_dp([3, 6, 5, 1, 8])

khởi tạo: dp = [0, -∞, -∞]
x= 3 (dư 0): dp[0]+3=3 -> new[0]  => dp = [3, -∞, -∞]
x= 6 (dư 0): dp[0]+6=9 -> new[0]  => dp = [9, -∞, -∞]
x= 5 (dư 2): dp[0]+5=14 -> new[2]  => dp = [9, -∞, 14]
x= 1 (dư 1): dp[0]+1=10 -> new[1], dp[2]+1=15 -> new[0]  => dp = [15, 10, 14]
x= 8 (dư 2): dp[0]+8=23 -> new[2], dp[1]+8=18 -> new[0], dp[2]+8=22 -> new[1]  => dp = [18, 22, 23]
Đáp án: 18


18

In [17]:
def trace_greedy(nums):
    s = sum(nums)
    ones = sorted(x for x in nums if x % 3 == 1)
    twos = sorted(x for x in nums if x % 3 == 2)
    print(f"nums={nums}: tổng={s}, dư={s % 3}, dư1={ones}, dư2={twos}")
    if s % 3 == 0:
        print("  đã chia hết ->", s); return s
    single, pair = (ones, twos) if s % 3 == 1 else (twos, ones)
    opts = []
    if single:
        opts.append(("bỏ 1 số", single[0]))
    if len(pair) >= 2:
        opts.append(("bỏ 2 số", pair[0] + pair[1]))
    for name, v in opts:
        print(f"  phương án {name}: bỏ {v} -> {s - v}")
    best = s - min(v for _, v in opts)
    print("  chọn ->", best)
    return best

trace_greedy([1, 2, 3, 4, 4]); trace_greedy([2, 2, 4, 7])

nums=[1, 2, 3, 4, 4]: tổng=14, dư=2, dư1=[1, 4, 4], dư2=[2]
  phương án bỏ 1 số: bỏ 2 -> 12
  phương án bỏ 2 số: bỏ 5 -> 9
  chọn -> 12
nums=[2, 2, 4, 7]: tổng=15, dư=0, dư1=[4, 7], dư2=[2, 2]
  đã chia hết -> 15


15

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def maxSumDivThree(self, nums: list[int]) -> int:
        # dp[r] = tổng lớn nhất của một tập con có tổng % 3 == r (-inf nếu chưa đạt được)
        dp = [0, float("-inf"), float("-inf")]
        for x in nums:
            new = dp[:]                       # tính từ bản CŨ để mỗi số dùng tối đa 1 lần
            for r in range(3):
                nr = (r + x) % 3
                new[nr] = max(new[nr], dp[r] + x)
            dp = new
        return dp[0]

for nums, want in examples:
    got = Solution().maxSumDivThree(nums)
    print(got, "OK" if got == want else "SAI")

18 OK
0 OK
12 OK


## 10. Vì sao thuật toán đúng

**DP.** Quy nạp theo số phần tử đã xét. Ban đầu chỉ có tập rỗng (tổng 0, dư 0) ⇒ `dp = [0, -∞, -∞]` đúng. Giả sử `dp` đúng với `nums[:i]`. Mọi tập con của `nums[:i+1]` hoặc không chứa `x = nums[i]` (được tính bởi `new = dp[:]`), hoặc chứa `x` — khi đó bỏ `x` đi được một tập con của `nums[:i]` dư `r`, và tập tốt nhất như vậy có tổng `dp[r]` ⇒ ứng viên `dp[r] + x` cho dư `(r + x) % 3`. Lấy max hai trường hợp ⇒ `new` đúng. Vì tính từ `dp` cũ, `x` không bị cộng hai lần.

**Tham lam.** Gọi `S` = tổng, giả sử `S ≡ 1`. Một tập tối ưu có phần bị **bỏ ra** `D` với `sum(D) ≡ 1` và `sum(D)` nhỏ nhất.
- Nếu `D` chứa một số dư 1 thì `{số dư 1 nhỏ nhất}` đã là phần bỏ hợp lệ có tổng ≤ `sum(D)` (vì mọi số dương).
- Nếu `D` không chứa số dư 1 nào thì `D` gồm số dư 0 và số dư 2, với số lượng số dư 2 ≡ 2 (mod 3) ⇒ ít nhất 2 số dư 2 ⇒ `{hai số dư 2 nhỏ nhất}` hợp lệ và có tổng ≤ `sum(D)`.

Vậy phần bỏ tối ưu luôn là một trong hai phương án ⇒ tham lam đúng. Trường hợp `S ≡ 2` đối xứng. ∎

## 11. Phân tích Time / Space

Gọi `n = len(nums)` ≤ 4·10⁴.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(2^n · n)` | `O(1)` |
| 2. Tham lam + sort | `O(n log n)` | `O(n)` |
| 3. Tham lam 2 nhỏ nhất | `O(n)` | `O(1)` |
| 4. DP bảng | `O(n)` | `O(n)` |
| 5. DP `dp[3]` | `O(n)` | `O(1)` |
| (+) Đệ quy có nhớ | `O(n)` | `O(n)` + đệ quy sâu `n` |

Không có khác biệt average/worst: mọi cách 2–5 duyệt mảng đúng một lần (cộng sắp xếp ở Cách 2). Tổng lớn nhất ≤ 4·10⁸ — Python không lo tràn số.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {
    "greedy_sort": max_sum_greedy_sort,
    "greedy_linear": max_sum_greedy_linear,
    "greedy_submit": lambda a: SolutionGreedy().maxSumDivThree(a),
    "dp_table": max_sum_dp_table,
    "dp": max_sum_dp,
    "dp_submit": lambda a: SolutionDP().maxSumDivThree(a),
    "memo": max_sum_memo,
    "submit": lambda a: Solution().maxSumDivThree(a),
}
edge = [
    ([1], 0), ([2], 0), ([3], 3),
    ([1, 1], 0), ([2, 2], 0), ([1, 2], 3),
    ([2, 2, 2], 6),                 # ba số dư 2
    ([1, 1, 1, 1], 3),              # tổng 4 dư 1 -> bỏ một số 1
    ([2, 2, 4, 7], 15),             # tổng đã chia hết
    ([17, 2, 2, 10], 27),           # bỏ hai số dư 2 tốt hơn bỏ một số dư 1
    ([3, 2, 2], 3),                 # không có số dư 1 -> buộc bỏ hai số dư 2
]
for nums, want in edge:
    assert max_sum_brute(nums) == want, (nums, max_sum_brute(nums), want)
for name, f in ALL.items():
    for nums, want in examples + edge:
        assert f(nums) == want, (name, nums, f(nums), want)
print("Mọi cách đúng với", len(examples + edge), "test (edge case đã kiểm chứng bằng brute force)")

Mọi cách đúng với 14 test (edge case đã kiểm chứng bằng brute force)


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

`n ≤ 14` để brute force chạy nhanh; giá trị trong `[1, 10^4]` và cả giá trị nhỏ `[1, 10]` (nhiều trùng lặp, nhiều trường hợp hoà).

In [20]:
rng = random.Random(1262)
for _ in range(1500):
    n = rng.randint(1, 14)
    hi = rng.choice([10, 10**4])
    nums = [rng.randint(1, hi) for _ in range(n)]
    want = max_sum_brute(nums)
    for name, f in ALL.items():
        got = f(nums)
        assert got == want, (name, nums, got, want)
print("1500 mảng ngẫu nhiên: mọi cách khớp brute force")

1500 mảng ngẫu nhiên: mọi cách khớp brute force


### 12.3 Giới hạn `n = 4·10^4`

(Bỏ qua `memo` vì đệ quy sâu `n` tầng.)

In [21]:
big = [rng.randint(1, 10**4) for _ in range(4 * 10**4)]
ref = None
for name in ["greedy_sort", "greedy_linear", "greedy_submit", "dp_table", "dp", "dp_submit", "submit"]:
    t0 = time.perf_counter()
    res = ALL[name](big)
    dt = time.perf_counter() - t0
    ref = res if ref is None else ref
    assert res == ref
    print(f"{name:14}: {res}  ({dt:.3f}s)")
assert ref % 3 == 0

greedy_sort   : 199852338  (0.008s)
greedy_linear : 199852338  (0.003s)
greedy_submit : 199852338  (0.003s)
dp_table      : 199852338  (0.031s)
dp            : 199852338  (0.021s)
dp_submit     : 199852338  (0.026s)
submit        : 199852338  (0.026s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam chỉ xét bỏ **một** phần tử

Quên phương án "bỏ hai số có dư kia". Có hai kiểu thất bại: (a) không có số nào cùng dư với tổng ⇒ bản sai bó tay trả `0`; (b) có, nhưng bỏ hai số dư kia **rẻ hơn**.

In [22]:
def greedy_wrong1(nums):
    s = sum(nums)
    r = s % 3
    if r == 0:
        return s
    same = [x for x in nums if x % 3 == r]
    return s - min(same) if same else 0          # SAI: quên phương án bỏ 2 số

for c in [[3, 2, 2], [17, 2, 2, 10]]:
    print(c, "tổng", sum(c), "dư", sum(c) % 3, "| sai:", greedy_wrong1(c), "| đúng:", Solution().maxSumDivThree(c))

[3, 2, 2] tổng 7 dư 1 | sai: 0 | đúng: 3
[17, 2, 2, 10] tổng 31 dư 1 | sai: 21 | đúng: 27


- `[3, 2, 2]`: tổng 7 dư 1, không có số dư 1 ⇒ bản sai trả 0; đúng ra bỏ `2 + 2` ⇒ còn **3**.
- `[17, 2, 2, 10]`: tổng 31 dư 1. Bản sai bỏ số dư 1 nhỏ nhất là `10` ⇒ 21; bỏ `2 + 2 = 4` ⇒ **27** tốt hơn.

**Sửa:** luôn so sánh **cả hai** phương án.

### Lỗi 2 — Cập nhật `dp` tại chỗ (dùng một phần tử hai lần)

Viết `dp[(r + x) % 3] = max(..., dp[r] + x)` trực tiếp trên `dp`: trong cùng một vòng `x`, giá trị vừa cập nhật lại được dùng ở `r` sau ⇒ `x` bị cộng nhiều lần.

In [23]:
def dp_wrong2(nums):
    NEG = float("-inf")
    dp = [0, NEG, NEG]
    for x in nums:
        for r in range(3):
            nr = (r + x) % 3
            dp[nr] = max(dp[nr], dp[r] + x)       # SAI: không dùng bản sao
    return dp[0]

for nums in [[1], [4], [2, 5]]:
    print(nums, "| sai:", dp_wrong2(nums), "| đúng:", Solution().maxSumDivThree(nums))

[1] | sai: 3 | đúng: 0
[4] | sai: 12 | đúng: 0
[2, 5] | sai: 9 | đúng: 0


Với `[1]`: `r=0` ⇒ `dp[1] = 1`; `r=1` ⇒ `dp[2] = 1 + 1 = 2` (dùng `1` lần hai!); `r=2` ⇒ `dp[0] = 2 + 1 = 3` ⇒ trả 3, trong khi đáp án là 0.

### Lỗi 3 — Khởi tạo `dp = [0, 0, 0]`

`dp[1] = 0` nghĩa là "có một tập dư 1 với tổng 0" — không tồn tại. Cộng thêm số dư 2 vào sẽ tạo ra một "tập dư 0" giả.

In [24]:
def dp_wrong3(nums):
    dp = [0, 0, 0]                               # SAI: trạng thái 1, 2 chưa đạt được
    for x in nums:
        new = dp[:]
        for r in range(3):
            nr = (r + x) % 3
            new[nr] = max(new[nr], dp[r] + x)
        dp = new
    return dp[0]

for nums in [[2], [4], [1, 1], [3, 6, 5, 1, 8]]:
    print(nums, "| sai:", dp_wrong3(nums), "| đúng:", Solution().maxSumDivThree(nums))

[2] | sai: 2 | đúng: 0
[4] | sai: 4 | đúng: 0
[1, 1] | sai: 2 | đúng: 0
[3, 6, 5, 1, 8] | sai: 23 | đúng: 18


### Lỗi 4 — `min()` trên danh sách rỗng / chỉ số `[1]` không tồn tại

Tham lam viết `min(ones[0], twos[0] + twos[1])` mà không kiểm tra đủ phần tử ⇒ `IndexError`. Ta bắt lỗi để minh hoạ (notebook không dừng).

In [25]:
def greedy_wrong4(nums):
    s = sum(nums)
    ones = sorted(x for x in nums if x % 3 == 1)
    twos = sorted(x for x in nums if x % 3 == 2)
    if s % 3 == 1:
        return s - min(ones[0], twos[0] + twos[1])   # SAI: có thể thiếu phần tử
    if s % 3 == 2:
        return s - min(twos[0], ones[0] + ones[1])
    return s

for nums in [[4], [3, 6, 5, 1, 8]]:
    try:
        print(nums, "| sai:", greedy_wrong4(nums))
    except IndexError as e:
        print(nums, "| sai: IndexError ->", e)
    print("      đúng:", Solution().maxSumDivThree(nums))

[4] | sai: IndexError -> list index out of range
      đúng: 0
[3, 6, 5, 1, 8] | sai: IndexError -> list index out of range
      đúng: 18


**Sửa:** chỉ thêm phương án khi đủ phần tử (Cách 2), hoặc dùng `INF` cho phần tử thiếu (Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"tổng lớn nhất / số cách sao cho tổng chia hết cho k"** ⇒ DP theo số dư `dp[r]`, `r ∈ [0, k)`. Độ phức tạp `O(n·k)`.
- `k` nhỏ (3) ⇒ tham lam "lấy hết rồi bỏ ít nhất" thường cho lời giải `O(n)`, `O(1)` rất gọn.
- DP knapsack 0/1 theo từng phần tử: **luôn cập nhật từ bản cũ** (`new = dp[:]`) — hoặc duyệt ngược nếu là knapsack theo trọng lượng.
- Trạng thái chưa đạt ⇒ `-inf` (bài max) hoặc `inf` (bài min) hoặc `0` (bài đếm số cách, trừ trạng thái gốc = 1).
- Kiểm tra nhanh: `[1]`, `[1, 1]`, `[2, 2, 2]`, `[17, 2, 2, 10]`.

## 15. Biến thể của bài

1. **Chia hết cho `k` tổng quát** ⇒ `dp` kích thước `k`.
2. **Đếm số tập con** có tổng chia hết cho 3 (modulo `10^9+7`) ⇒ `dp[r]` là số cách.
3. **Mảng con liên tiếp** có tổng chia hết cho `k` (bài 974, 523) ⇒ prefix sum + đếm số dư — khác hẳn vì phải liên tiếp.
4. **LeetCode 1363 — Largest Multiple of Three**: chọn chữ số để ghép thành số lớn nhất chia hết cho 3 — đúng tư duy tham lam bỏ 1 hoặc 2 chữ số nhỏ nhất.
5. **Tổng lớn nhất với số phần tử chọn ≤ m** ⇒ DP thêm chiều số lượng.

Code biến thể 1 & 2:

In [26]:
def max_sum_div_k(nums, k):
    NEG = float("-inf")
    dp = [0] + [NEG] * (k - 1)
    for x in nums:
        new = dp[:]
        for r in range(k):
            if dp[r] != NEG:
                nr = (r + x) % k
                new[nr] = max(new[nr], dp[r] + x)
        dp = new
    return dp[0]

def count_subsets_div3(nums, mod=10**9 + 7):
    cnt = [1, 0, 0]                       # tập rỗng
    for x in nums:
        new = cnt[:]
        for r in range(3):
            new[(r + x) % 3] = (new[(r + x) % 3] + cnt[r]) % mod
        cnt = new
    return cnt[0]

print(max_sum_div_k([3, 6, 5, 1, 8], 3), max_sum_div_k([3, 6, 5, 1, 8], 5), max_sum_div_k([3, 6, 5, 1, 8], 7))
print("Số tập con chia hết cho 3 của [1,2,3]:", count_subsets_div3([1, 2, 3]))

18 20 14
Số tập con chia hết cho 3 của [1,2,3]: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Thử mọi tập con | `O(2^n·n)` | `O(1)` | Kiểm chứng, `n ≤ 20` |
| 2. Tham lam + sort | Tổng − min(bỏ 1 số cùng dư, bỏ 2 số dư kia) | `O(n log n)` | `O(n)` | Dễ giải thích |
| 3. Tham lam 2 nhỏ nhất | Như 2 nhưng chỉ giữ 4 biến | `O(n)` | `O(1)` | Tối ưu nhất cho `k = 3` |
| 4. DP bảng | `dp[i][r]` chọn/không chọn | `O(n)` | `O(n)` | Học DP, cần truy vết tập được chọn |
| 5. DP `dp[3]` | Thu gọn còn 3 trạng thái | `O(n)` | `O(1)` | Mặc định, tổng quát cho mọi `k` |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().maxSumDivThree([2, 6, 2, 2, 7])` bằng bao nhiêu?

### Bài 2 (tính tay)
Chạy tay DP `dp[3]` với `nums = [4, 1, 5, 3, 1]`, ghi `dp` sau mỗi phần tử.

### Bài 3 (điền TODO)
Hoàn thiện `max_sum_div_k_greedy_check(nums, k)` — hàm **DP** cho số chia `k` bất kỳ bằng **bảng 2 chiều** (để tiện truy vết).

### Bài 4 (tự viết lại)
Viết lại tham lam bằng `heapq.nsmallest`: `max_sum_heap(nums)`.

### Bài 5 (mini challenge)
Viết `best_subset(nums)`: trả về **một tập chỉ số** đạt tổng lớn nhất chia hết cho 3 (truy vết DP).

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_sum_div_k_table(nums, k):
    NEG = float("-inf")
    dp = [[NEG] * k for _ in range(len(nums) + 1)]
    dp[0][0] = 0
    # TODO: điền dp[i][r] = max(không chọn, chọn)
    # TODO: trả về dp[n][0]
    pass

# Bài 4 — TODO
def max_sum_heap(nums):
    # TODO: dùng heapq.nsmallest(2, ...) cho từng loại dư
    pass

# Bài 5 — TODO
def best_subset(nums):
    # TODO: dựng bảng dp[i][r], rồi đi ngược từ (n, 0) để biết phần tử nào được chọn
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tổng = 19, dư 1. (2) Số dư 1: `7`; số dư 2: `2, 2, 2`. (3) Bỏ `7` ⇒ 12, bỏ `2+2` ⇒ 15 ⇒ đáp án 15.

**Bài 2.** (1) Bắt đầu `[0, -∞, -∞]`. (2) Sau 4: `[0, 4, -∞]`; sau 1: `[0, 4, 5]`; sau 5: `[9, 10, 5]`. (3) Sau 3: `[12, 13, 8]`; sau 1: `[12, 13, 14]` ⇒ đáp án 12 (kiểm tra bằng tham lam: tổng 14 dư 2, bỏ `1 + 1` rẻ hơn bỏ `5`).

**Bài 3.** (1) `skip = dp[i-1][r]`. (2) `take = dp[i-1][(r - x) % k] + x`. (3) `dp[i][r] = max(skip, take)`.

**Bài 4.** (1) `heapq.nsmallest(2, (x for x in nums if x % 3 == 1))`. (2) Phương án chỉ hợp lệ nếu đủ phần tử. (3) So khớp với `Solution` trên test ngẫu nhiên.

**Bài 5.** (1) Lưu cả bảng. (2) Ở bước `i`, nếu `dp[i][r] == dp[i-1][r]` thì có thể không chọn `nums[i-1]`; ngược lại chọn và chuyển sang `r = (r - x) % 3`. (3) Kiểm tra tổng các phần tử được chọn bằng đáp án.

## 19. Đáp án tham khảo

In [28]:
import heapq

# Bài 1
assert Solution().maxSumDivThree([2, 6, 2, 2, 7]) == 15
# Bài 2
assert Solution().maxSumDivThree([4, 1, 5, 3, 1]) == 12

# Bài 3
def max_sum_div_k_table_sol(nums, k):
    NEG = float("-inf")
    n = len(nums)
    dp = [[NEG] * k for _ in range(n + 1)]
    dp[0][0] = 0
    for i in range(1, n + 1):
        x = nums[i - 1]
        for r in range(k):
            dp[i][r] = max(dp[i - 1][r], dp[i - 1][(r - x) % k] + x)
    return dp[n][0]

# Bài 4
def max_sum_heap_sol(nums):
    s = sum(nums)
    r = s % 3
    if r == 0:
        return s
    ones = heapq.nsmallest(2, (x for x in nums if x % 3 == 1))
    twos = heapq.nsmallest(2, (x for x in nums if x % 3 == 2))
    single, pair = (ones, twos) if r == 1 else (twos, ones)
    opts = []
    if single:
        opts.append(single[0])
    if len(pair) == 2:
        opts.append(sum(pair))
    return s - min(opts)

# Bài 5
def best_subset_sol(nums):
    NEG = float("-inf")
    n = len(nums)
    dp = [[NEG] * 3 for _ in range(n + 1)]
    dp[0][0] = 0
    for i in range(1, n + 1):
        x = nums[i - 1]
        for r in range(3):
            dp[i][r] = max(dp[i - 1][r], dp[i - 1][(r - x) % 3] + x)
    chosen, r = [], 0
    for i in range(n, 0, -1):
        if dp[i][r] == dp[i - 1][r]:
            continue                       # không chọn nums[i-1]
        chosen.append(i - 1)
        r = (r - nums[i - 1]) % 3
    return sorted(chosen)

rng = random.Random(31)
for _ in range(500):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 12))]
    want = max_sum_brute(nums)
    assert max_sum_div_k_table_sol(nums, 3) == want
    assert max_sum_heap_sol(nums) == want
    idx = best_subset_sol(nums)
    assert len(set(idx)) == len(idx) and sum(nums[i] for i in idx) == want and want % 3 == 0
    k = rng.randint(1, 7)
    assert max_sum_div_k_table_sol(nums, k) == max_sum_div_k(nums, k)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP theo số dư:**

```python
NEG = float('-inf')
dp = [0] + [NEG] * (k - 1)          # dp[r] = tổng lớn nhất có dư r
for x in nums:
    new = dp[:]                     # bản sao: mỗi phần tử dùng ≤ 1 lần
    for r in range(k):
        nr = (r + x) % k
        new[nr] = max(new[nr], dp[r] + x)
    dp = new
return dp[0]
```

**Template tham lam (k = 3):** `S − min(bỏ 1 số dư S%3 nhỏ nhất, bỏ 2 số dư (3 − S%3) nhỏ nhất)`.

**Checklist trước khi submit:**
- [ ] Trạng thái chưa đạt khởi tạo `-inf`, chỉ `dp[0] = 0`.
- [ ] Cập nhật từ **bản cũ** (`new = dp[:]`).
- [ ] Tham lam xét **cả hai** phương án bỏ 1 số và bỏ 2 số.
- [ ] Kiểm tra đủ phần tử trước khi lấy `[0]`, `[1]`.
- [ ] Không chọn gì ⇒ 0 (ví dụ `[4]`).
- [ ] Đúng chữ ký: `maxSumDivThree(self, nums: list[int]) -> int`.